# 01 — Exploración de MNIST como grafo de píxeles

Objetivo: documentar fuente, dimensiones, balance de clases, intensidades y estructura espacial antes de modelar.

In [ ]:
from pathlib import Path
import sys, urllib.request
import numpy as np

SEED = 2026
rng = np.random.default_rng(SEED)

# Funciona desde notebooks/ en el repositorio y también en Colab.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "raw" / "mnist.npz"
SRC = ROOT / "src"
if not SRC.exists():
    # Si se subió solo el notebook, crea una ruta local y descarga los datos.
    ROOT = Path.cwd()
    DATA = ROOT / "mnist.npz"
if not DATA.exists():
    DATA.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz", DATA
    )
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

data = np.load(DATA)
x_train = data["x_train"].astype(np.float32) / 255.0
y_train = data["y_train"].astype(np.int64)
x_test = data["x_test"].astype(np.float32) / 255.0
y_test = data["y_test"].astype(np.int64)
print("Semilla:", SEED, "| entrenamiento:", x_train.shape, "| prueba:", x_test.shape)

## Diccionario de variables

| Variable | Tipo | Unidad | Descripción |
|---|---|---|---|
| `x_train`, `x_test` | uint8 → float32 | intensidad [0,1] | Imágenes 28×28 |
| `y_train`, `y_test` | entero | clase 0–9 | Dígito real |

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for k, ax in enumerate(axes.ravel()):
    idx = np.flatnonzero(y_train == k)[0]
    ax.imshow(x_train[idx], cmap="gray")
    ax.set_title(f"Clase {k}")
    ax.axis("off")
plt.suptitle("Una observación por clase")
plt.tight_layout(); plt.show()

In [ ]:
counts = np.bincount(y_train, minlength=10)
plt.figure(figsize=(9, 4))
plt.bar(range(10), counts, color="#13A89E")
plt.xticks(range(10)); plt.xlabel("Clase"); plt.ylabel("Observaciones")
plt.title("Distribución de clases en entrenamiento")
for k, c in enumerate(counts): plt.text(k, c + 50, f"{c:,}", ha="center", fontsize=8)
plt.show()
print("Razón mayor/menor:", counts.max()/counts.min())

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(x_train[y_train == k].mean(axis=0), cmap="viridis")
    ax.set_title(f"Promedio {k}"); ax.axis("off")
plt.suptitle("Imagen promedio por clase")
plt.tight_layout(); plt.show()

## Lectura

Las clases son aproximadamente balanceadas (la clase mayor no duplica a la menor), por lo que se reporta exactitud junto con métricas macro. Los promedios conservan geometría local; por eso tiene sentido probar un modelo que utilice la vecindad de Moore entre píxeles.